# 26 - Multimodal Agent

A multimodal agent can **see or process images/documents and then act**.

Example:
```text
image
 ↓
vision model
 ↓
understand user goal
 ↓
choose tool
 ↓
tool result
 ↓
answer
```

This differs from multimodal RAG because the agent can dynamically choose actions based on visual evidence.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU openai pillow langchain langchain-openai

## 1. Create demo image

Use a local image so the notebook has a deterministic artifact.

In [ ]:
from PIL import Image, ImageDraw

img = Image.new('RGB', (700, 350), 'white')
draw = ImageDraw.Draw(img)
draw.rectangle((50, 50, 650, 300), outline='black', width=3)
draw.text((100, 120), 'Product: Demo Laptop', fill='black')
draw.text((100, 170), 'ID: P100', fill='black')
img.save('agent_product.png')

display(img)

## 2. Vision model input

A vision-capable model can receive the image as part of the request.

In [ ]:
import base64
from openai import OpenAI

if os.getenv('OPENAI_API_KEY') and os.getenv('OPENAI_VISION_MODEL'):
    client = OpenAI()

    with open('agent_product.png', 'rb') as f:
        image_b64 = base64.b64encode(f.read()).decode('utf-8')

    response = client.responses.create(
        model=os.environ['OPENAI_VISION_MODEL'],
        input=[{
            'role': 'user',
            'content': [
                {'type': 'input_text', 'text': 'Read the product ID from this image.'},
                {'type': 'input_image', 'image_url': f'data:image/png;base64,{image_b64}'},
            ],
        }],
    )

    print(response.output_text)
else:
    print('Set OPENAI_API_KEY and OPENAI_VISION_MODEL for the live vision example.')

## 3. Tool based on visual result

The model can pass the detected ID into a tool.

In [ ]:
from langchain.tools import tool

@tool
def lookup_product(product_id: str) -> str:
    'Look up product facts by product ID.'
    catalog = {
        'P100': {'name': 'Demo Laptop', 'price': 900, 'ram_gb': 16}
    }
    return str(catalog.get(product_id, {'error': 'unknown product'}))

print(lookup_product.invoke({'product_id': 'P100'}))

## 4. Agent architecture

```text
image + user goal
      ↓
vision model
      ↓
tool selection
      ↓
tool call
      ↓
result
      ↓
final answer
```

## 5. Safety

Visual input can contain:
- hidden instructions
- malicious QR codes
- private information
- deceptive content

Treat image-derived text as untrusted input and keep tools permissioned.

## 6. Evaluation and practice

Measure:
- visual extraction accuracy
- tool selection
- argument correctness
- task success
- hallucinated visual facts

Practice: build a product-image assistant that reads an ID and retrieves catalog facts.

## Evaluation

Improvement:
- image resolution/detail tuning
- OCR fallback
- structured visual extraction
- evidence linking
- human approval for external actions.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/